# 🤖 Agentic Post-Training Framework — Full Demo on A100

**Run this notebook on Google Colab with A100 GPU runtime.**

This notebook demonstrates the complete agentic post-training framework:

| Priority | Techniques | What You'll See |
|----------|-----------|----------------|
| ⭐ Core | **PPO, GRPO, DPO, SPO, RLHF** | Real training on GPT-2, live metrics, agent coordination |
| 🔷 Advanced | **KTO, ORPO, SimPO** | Quick demonstrations with comparison |
| 🔶 Optimization | **Quantization, Evaluation** | GPTQ 4-bit, benchmark results |

**Architecture:** Specialized agents (Coordinator, Trainer, Optimizer, Evaluator) communicate via a message bus to orchestrate the full pipeline.

---

⚡ **Runtime:** Go to Runtime → Change runtime type → **A100 GPU**

## 0. Setup & Hardware Check

In [ ]:
%%time
# Install dependencies
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes sentencepiece
!pip install -q tqdm matplotlib numpy pandas

In [ ]:
import torch
import gc
import time
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Any, Optional
from IPython.display import display, HTML, clear_output

# Hardware check
print("=" * 60)
print("🖥️  HARDWARE CHECK")
print("=" * 60)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu}")
    print(f"✅ VRAM: {mem:.1f} GB")
    print(f"✅ CUDA: {torch.version.cuda}")
    DEVICE = "cuda"
else:
    print("⚠️  No GPU detected — running in CPU mode (slower)")
    DEVICE = "cpu"

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ Device: {DEVICE}")
print("=" * 60)

# Seed for reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## 1. Agent Framework — Communication & Coordination

First, let's build the agent system. Each agent has a role, status, and communicates via a **message bus**.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# AGENT FRAMEWORK — Communication Bus & Agent Classes
# ═══════════════════════════════════════════════════════════════

from enum import Enum
from dataclasses import dataclass, field
import time

class AgentStatus(Enum):
    IDLE = "⚪ idle"
    RUNNING = "🔵 running"
    COMPLETED = "🟢 completed"
    FAILED = "🔴 failed"

class MessageType(Enum):
    COORDINATION = "🔗"
    STATUS = "📊"
    RESULT = "✅"
    OPTIMIZATION = "⚡"
    EVALUATION = "📈"

@dataclass
class Message:
    sender: str
    receiver: str
    msg_type: MessageType
    content: str
    timestamp: float = field(default_factory=time.time)

    def display(self):
        ts = time.strftime("%H:%M:%S", time.localtime(self.timestamp))
        colors = {
            "Coordinator": "#d2a8ff",
            "Trainer": "#79c0ff",
            "Optimizer": "#e3b341",
            "Evaluator": "#56d364",
        }
        color = colors.get(self.sender, "#c9d1d9")
        return f'<span style="color:#484f58">[{ts}]</span> {self.msg_type.value} <span style="color:{color};font-weight:600">{self.sender}</span> → {self.content}'


class MessageBus:
    """Central communication hub for all agents."""
    def __init__(self):
        self.history: list[Message] = []
        self._html_lines: list[str] = []

    def send(self, sender: str, receiver: str, msg_type: MessageType, content: str):
        msg = Message(sender, receiver, msg_type, content)
        self.history.append(msg)
        self._html_lines.append(msg.display())

    def show(self, last_n: int = 0):
        lines = self._html_lines[-last_n:] if last_n else self._html_lines
        html = '<div style="background:#0d1117;border:1px solid #30363d;border-radius:8px;padding:16px;font-family:monospace;font-size:13px;line-height:1.8;max-height:500px;overflow-y:auto">'
        html += '<div style="background:#161b22;margin:-16px -16px 12px;padding:8px 16px;border-radius:8px 8px 0 0">'
        html += '<span style="color:#ff5f57">●</span> <span style="color:#febc2e">●</span> <span style="color:#28c840">●</span>'
        html += ' <span style="color:#8b949e;font-size:12px">Agent Communication Log</span></div>'
        html += '<br>'.join(lines)
        html += '</div>'
        display(HTML(html))

    def summary_table(self):
        counts = {}
        for m in self.history:
            counts[m.sender] = counts.get(m.sender, 0) + 1
        html = '<table style="border-collapse:collapse;font-family:sans-serif">'
        html += '<tr style="border-bottom:2px solid #30363d"><th style="padding:8px 16px;text-align:left">Agent</th><th style="padding:8px 16px">Messages Sent</th></tr>'
        for agent, count in sorted(counts.items()):
            html += f'<tr><td style="padding:6px 16px">{agent}</td><td style="padding:6px 16px;text-align:center">{count}</td></tr>'
        html += f'<tr style="border-top:2px solid #30363d;font-weight:bold"><td style="padding:8px 16px">Total</td><td style="padding:8px 16px;text-align:center">{len(self.history)}</td></tr>'
        html += '</table>'
        display(HTML(html))


# Initialize the global message bus
bus = MessageBus()
print("✅ Agent framework initialized")
print("   MessageBus ready for inter-agent communication")

## 2. Load Model & Tokenizer

We'll use **GPT-2** for fast demonstration. On A100, you can swap to larger models.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

bus.send("Coordinator", "broadcast", MessageType.COORDINATION, "Initiating model loading...")

MODEL_NAME = "gpt2"  # Change to "gpt2-medium" or "gpt2-large" on A100

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
ref_model.eval()

num_params = sum(p.numel() for p in model.parameters()) / 1e6
bus.send("Trainer", "Coordinator", MessageType.STATUS, f"Model loaded: <b>{MODEL_NAME}</b> ({num_params:.0f}M params) on {DEVICE.upper()}")

bus.show()

## 3. Prepare Training Data

We'll create preference pairs (chosen/rejected) for alignment training.

In [ ]:
bus.send("Coordinator", "Trainer", MessageType.COORDINATION, "Starting data preparation stage...")

# Create synthetic preference data for demonstration
# In production, you'd use datasets like Anthropic-HH, UltraFeedback, etc.
PROMPTS = [
    "Explain quantum computing in simple terms:",
    "Write a Python function to sort a list:",
    "What is the meaning of life?",
    "How do neural networks learn?",
    "Describe the water cycle:",
    "What causes seasons on Earth?",
    "Explain how a compiler works:",
    "What is photosynthesis?",
    "How does encryption work?",
    "Explain gravity to a five year old:",
    "What makes a good leader?",
    "How do computers store data?",
    "What is machine learning?",
    "Explain the theory of relativity:",
    "How does the internet work?",
    "What is DNA?",
]

# Tokenize prompts
def tokenize_prompts(prompts, max_length=64):
    return tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=max_length
    ).to(DEVICE)

prompt_inputs = tokenize_prompts(PROMPTS)

# Generate chosen/rejected pairs using the model itself (for demo)
def generate_pairs(model, inputs, num_pairs=16):
    """Generate response pairs with different temperatures for preference data."""
    model.eval()
    pairs = []
    with torch.no_grad():
        # "Chosen" = lower temperature (more coherent)
        chosen_ids = model.generate(
            inputs["input_ids"][:num_pairs],
            attention_mask=inputs["attention_mask"][:num_pairs],
            max_new_tokens=48, temperature=0.7, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        # "Rejected" = higher temperature (less coherent)
        rejected_ids = model.generate(
            inputs["input_ids"][:num_pairs],
            attention_mask=inputs["attention_mask"][:num_pairs],
            max_new_tokens=48, temperature=1.5, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return chosen_ids, rejected_ids

chosen_ids, rejected_ids = generate_pairs(model, prompt_inputs)

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"Data prepared: {len(PROMPTS)} prompts, {len(chosen_ids)} preference pairs")
bus.show()

---

## 4. ⭐ Core Techniques — Real Training

### 4.1 DPO — Direct Preference Optimization

**Key Idea:** Instead of training a reward model + RL, DPO directly optimizes the policy with a simple loss on preference pairs.

$$\mathcal{L}_{\text{DPO}} = -\log \sigma \left( \beta \cdot \left[ \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right] \right)$$

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DPO — Direct Preference Optimization
# Paper: Rafailov et al., 2023
# Key: No reward model needed, simple BCE loss on preferences
# ═══════════════════════════════════════════════════════════════

import torch.nn.functional as F

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>DPO</b> training — Direct Preference Optimization")

def compute_log_probs(model, input_ids, labels):
    """Compute per-token log probabilities."""
    outputs = model(input_ids)
    logits = outputs.logits[:, :-1, :]
    target = labels[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(2, target.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.sum(dim=-1)  # Sum over sequence


def dpo_loss(model, ref_model, chosen_ids, rejected_ids, beta=0.1):
    """Compute DPO loss: -log σ(β * (log_ratio_chosen - log_ratio_rejected))"""
    # Policy log probs
    pi_chosen = compute_log_probs(model, chosen_ids, chosen_ids)
    pi_rejected = compute_log_probs(model, rejected_ids, rejected_ids)

    # Reference log probs
    with torch.no_grad():
        ref_chosen = compute_log_probs(ref_model, chosen_ids, chosen_ids)
        ref_rejected = compute_log_probs(ref_model, rejected_ids, rejected_ids)

    # DPO objective
    chosen_log_ratio = pi_chosen - ref_chosen
    rejected_log_ratio = pi_rejected - ref_rejected
    logits = beta * (chosen_log_ratio - rejected_log_ratio)

    loss = -F.logsigmoid(logits).mean()

    # Metrics
    with torch.no_grad():
        chosen_reward = beta * chosen_log_ratio.mean().item()
        rejected_reward = beta * rejected_log_ratio.mean().item()
        accuracy = (logits > 0).float().mean().item()

    return loss, {
        "loss": loss.item(),
        "chosen_reward": chosen_reward,
        "rejected_reward": rejected_reward,
        "reward_margin": chosen_reward - rejected_reward,
        "accuracy": accuracy,
    }


# Train DPO
dpo_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(dpo_model.parameters(), lr=5e-5, weight_decay=0.01)

DPO_STEPS = 50
dpo_metrics = []

bus.send("Trainer", "broadcast", MessageType.STATUS,
         f"Beginning DPO training for {DPO_STEPS} steps (β=0.1)")

dpo_model.train()
for step in range(1, DPO_STEPS + 1):
    # Mini-batch (cycle through data)
    idx = torch.randint(0, len(chosen_ids), (4,))
    loss, metrics = dpo_loss(
        dpo_model, ref_model,
        chosen_ids[idx], rejected_ids[idx],
        beta=0.1
    )

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(dpo_model.parameters(), 1.0)
    optimizer.step()

    dpo_metrics.append(metrics)

    if step % 10 == 0:
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"DPO Step {step}/{DPO_STEPS} | Loss: <b>{metrics['loss']:.4f}</b> | "
                 f"Acc: {metrics['accuracy']:.2f} | Margin: {metrics['reward_margin']:.4f}")

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"DPO training complete! Final loss: <b>{dpo_metrics[-1]['loss']:.4f}</b>, "
         f"Accuracy: <b>{dpo_metrics[-1]['accuracy']:.2f}</b>")
bus.show()

In [ ]:
# 📊 DPO Training Curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("DPO Training Results", fontsize=14, fontweight="bold")

steps = range(1, len(dpo_metrics) + 1)

axes[0].plot(steps, [m["loss"] for m in dpo_metrics], color="#7c3aed", linewidth=2)
axes[0].set_title("Loss"); axes[0].set_xlabel("Step"); axes[0].grid(alpha=0.3)

axes[1].plot(steps, [m["accuracy"] for m in dpo_metrics], color="#06b6d4", linewidth=2)
axes[1].set_title("Preference Accuracy"); axes[1].set_xlabel("Step"); axes[1].grid(alpha=0.3)

axes[2].plot(steps, [m["chosen_reward"] for m in dpo_metrics], color="#34d399", linewidth=2, label="Chosen")
axes[2].plot(steps, [m["rejected_reward"] for m in dpo_metrics], color="#f87171", linewidth=2, label="Rejected")
axes[2].set_title("Implicit Rewards"); axes[2].set_xlabel("Step"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 GRPO — Group Relative Policy Optimization

**Key Idea (DeepSeek-R1):** Generate a GROUP of responses per prompt, rank them, use group statistics as advantages. **No value model needed** — saves ~50% memory vs PPO.

$$A_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r}) + \epsilon}$$

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GRPO — Group Relative Policy Optimization (DeepSeek-R1)
# Paper: Shao et al., 2024
# Key: No value model! Group-level advantages from multiple samples
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>GRPO</b> training — Group Relative Policy Optimization (DeepSeek-R1)")

# Clean up DPO model to free memory
del dpo_model, optimizer
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


def simple_reward(text):
    """Simple reward function based on response quality heuristics."""
    score = 0.0
    # Reward coherence (penalize repetition)
    words = text.split()
    if len(words) > 3:
        unique_ratio = len(set(words)) / len(words)
        score += unique_ratio * 2.0
    # Reward length (but not too long)
    score += min(len(words) / 20.0, 1.5)
    # Penalize empty or very short
    if len(words) < 3:
        score -= 1.0
    return score


def grpo_step(model, ref_model, prompt_ids, prompt_mask, group_size=4,
              max_new_tokens=32, beta_kl=0.1, clip_ratio=0.2):
    """Single GRPO step: sample group, compute advantages, policy gradient."""
    model.eval()

    # 1. Generate GROUP of responses for each prompt
    all_responses = []
    all_rewards = []
    batch_size = prompt_ids.shape[0]

    with torch.no_grad():
        for g in range(group_size):
            gen_ids = model.generate(
                prompt_ids, attention_mask=prompt_mask,
                max_new_tokens=max_new_tokens,
                temperature=0.8 + 0.1 * g,  # Vary temperature across group
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
            all_responses.append(gen_ids)

            # Score responses
            texts = tokenizer.batch_decode(gen_ids[:, prompt_ids.shape[1]:], skip_special_tokens=True)
            rewards = torch.tensor([simple_reward(t) for t in texts], device=DEVICE)
            all_rewards.append(rewards)

    # 2. Compute group-relative advantages
    reward_stack = torch.stack(all_rewards, dim=0)  # [group_size, batch]
    mean_reward = reward_stack.mean(dim=0, keepdim=True)
    std_reward = reward_stack.std(dim=0, keepdim=True) + 1e-8
    advantages = (reward_stack - mean_reward) / std_reward  # [group_size, batch]

    # 3. Compute policy gradient with advantages (on best response per group)
    model.train()
    best_idx = reward_stack.argmax(dim=0)  # [batch]

    # Get the best response for each prompt
    best_responses = torch.stack([
        all_responses[best_idx[b].item()][b]
        for b in range(batch_size)
    ])
    best_advantages = advantages[best_idx, torch.arange(batch_size)]

    # Forward pass for policy loss
    outputs = model(best_responses)
    logits = outputs.logits[:, :-1, :]
    targets = best_responses[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_lp = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)
    seq_lp = token_lp.sum(dim=-1)

    # Reference model KL
    with torch.no_grad():
        ref_out = ref_model(best_responses)
        ref_logits = ref_out.logits[:, :-1, :]
        ref_lp = F.log_softmax(ref_logits, dim=-1)
        ref_token_lp = ref_lp.gather(2, targets.unsqueeze(-1)).squeeze(-1)
        ref_seq_lp = ref_token_lp.sum(dim=-1)

    kl = (seq_lp - ref_seq_lp).mean()

    # GRPO loss: weighted by group advantage, no value loss!
    policy_loss = -(seq_lp * best_advantages.detach()).mean()
    loss = policy_loss + beta_kl * kl

    metrics = {
        "loss": loss.item(),
        "policy_loss": policy_loss.item(),
        "kl": kl.item(),
        "mean_reward": reward_stack.mean().item(),
        "reward_std": reward_stack.std().item(),
        "best_reward": reward_stack.max(dim=0).values.mean().item(),
        "advantage_mean": advantages.mean().item(),
    }
    return loss, metrics


# Train GRPO
grpo_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(grpo_model.parameters(), lr=1e-5, weight_decay=0.01)

GRPO_STEPS = 30
GROUP_SIZE = 4
grpo_metrics = []

bus.send("Trainer", "broadcast", MessageType.STATUS,
         f"Beginning GRPO training: {GRPO_STEPS} steps, group_size={GROUP_SIZE}")

for step in range(1, GRPO_STEPS + 1):
    # Sample batch of prompts
    idx = torch.randint(0, len(PROMPTS), (4,))
    batch_ids = prompt_inputs["input_ids"][idx]
    batch_mask = prompt_inputs["attention_mask"][idx]

    loss, metrics = grpo_step(
        grpo_model, ref_model, batch_ids, batch_mask,
        group_size=GROUP_SIZE
    )

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(grpo_model.parameters(), 1.0)
    optimizer.step()

    grpo_metrics.append(metrics)

    if step % 5 == 0:
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"GRPO Step {step}/{GRPO_STEPS} | Loss: <b>{metrics['loss']:.4f}</b> | "
                 f"Reward: {metrics['mean_reward']:.3f} | KL: {metrics['kl']:.4f}")

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"GRPO complete! Final reward: <b>{grpo_metrics[-1]['best_reward']:.3f}</b>, "
         f"KL: {grpo_metrics[-1]['kl']:.4f}")
bus.show()

In [ ]:
# 📊 GRPO Training Curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("GRPO Training Results (DeepSeek-R1 Technique)", fontsize=14, fontweight="bold")

steps = range(1, len(grpo_metrics) + 1)

axes[0].plot(steps, [m["loss"] for m in grpo_metrics], color="#7c3aed", linewidth=2)
axes[0].set_title("Loss"); axes[0].set_xlabel("Step"); axes[0].grid(alpha=0.3)

axes[1].plot(steps, [m["mean_reward"] for m in grpo_metrics], color="#3b82f6", linewidth=2, label="Mean")
axes[1].plot(steps, [m["best_reward"] for m in grpo_metrics], color="#34d399", linewidth=2, label="Best in group")
axes[1].set_title("Group Rewards"); axes[1].set_xlabel("Step"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(steps, [m["reward_std"] for m in grpo_metrics], color="#f59e0b", linewidth=2)
axes[2].set_title("Group Reward Std (should decrease)"); axes[2].set_xlabel("Step"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3 PPO — Proximal Policy Optimization

**Key Idea:** The backbone of RLHF. Optimize a clipped surrogate objective with a value head and reward model.

$$\mathcal{L}^{PPO} = -\min(r_t A_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t) + c_1 \mathcal{L}^{VF} - c_2 S$$

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PPO — Proximal Policy Optimization for RLHF
# Paper: Schulman et al., 2017
# Key: Clipped surrogate + value head + GAE advantages
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>PPO</b> training — Proximal Policy Optimization")

del grpo_model, optimizer
gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

import torch.nn as nn

class PolicyWithValueHead(nn.Module):
    """GPT-2 with a value head for PPO."""
    def __init__(self, model_name):
        super().__init__()
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.value_head = nn.Linear(self.model.config.n_embd, 1)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
        logits = outputs.logits
        hidden = outputs.hidden_states[-1]
        values = self.value_head(hidden).squeeze(-1)
        return logits, values

    def generate(self, *args, **kwargs):
        return self.model.generate(*args, **kwargs)


def ppo_step(policy, ref_model, prompt_ids, prompt_mask,
             clip_ratio=0.2, value_coef=0.5, entropy_coef=0.01, kl_coef=0.1):
    """Single PPO step with clipped surrogate objective."""

    # Generate responses
    policy.eval()
    with torch.no_grad():
        gen_ids = policy.generate(
            prompt_ids, attention_mask=prompt_mask,
            max_new_tokens=32, temperature=0.8, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        # Old policy log probs and values
        old_logits, old_values = policy(gen_ids)
        old_log_probs = F.log_softmax(old_logits[:, :-1, :], dim=-1)
        old_token_lp = old_log_probs.gather(2, gen_ids[:, 1:].unsqueeze(-1)).squeeze(-1)

    # Rewards
    texts = tokenizer.batch_decode(gen_ids[:, prompt_ids.shape[1]:], skip_special_tokens=True)
    rewards = torch.tensor([simple_reward(t) for t in texts], device=DEVICE)

    # Simple advantage: reward - value (no GAE for brevity)
    response_start = prompt_ids.shape[1]
    values_at_end = old_values[:, -1].detach()
    advantages = rewards - values_at_end
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    returns = rewards

    # PPO update
    policy.train()
    new_logits, new_values = policy(gen_ids)
    new_log_probs = F.log_softmax(new_logits[:, :-1, :], dim=-1)
    new_token_lp = new_log_probs.gather(2, gen_ids[:, 1:].unsqueeze(-1)).squeeze(-1)

    # Sequence-level ratio
    new_seq_lp = new_token_lp.sum(dim=-1)
    old_seq_lp = old_token_lp.sum(dim=-1)
    ratio = torch.exp(new_seq_lp - old_seq_lp)

    # Clipped surrogate
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()

    # Value loss
    value_loss = F.mse_loss(new_values[:, -1], returns)

    # Entropy bonus
    probs = F.softmax(new_logits[:, -1, :], dim=-1)
    entropy = -(probs * (probs + 1e-8).log()).sum(dim=-1).mean()

    # KL from reference
    with torch.no_grad():
        ref_out = ref_model(gen_ids)
        ref_lp = F.log_softmax(ref_out.logits[:, :-1, :], dim=-1)
        ref_token_lp = ref_lp.gather(2, gen_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
        kl = (new_token_lp - ref_token_lp).sum(dim=-1).mean()

    loss = policy_loss + value_coef * value_loss - entropy_coef * entropy + kl_coef * kl

    metrics = {
        "loss": loss.item(),
        "policy_loss": policy_loss.item(),
        "value_loss": value_loss.item(),
        "entropy": entropy.item(),
        "kl": kl.item(),
        "reward": rewards.mean().item(),
        "clip_fraction": ((ratio - 1.0).abs() > clip_ratio).float().mean().item(),
    }
    return loss, metrics


# Train PPO
ppo_policy = PolicyWithValueHead(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(ppo_policy.parameters(), lr=1e-5)

PPO_STEPS = 30
ppo_metrics = []

bus.send("Trainer", "broadcast", MessageType.STATUS,
         f"Beginning PPO training: {PPO_STEPS} steps (clip=0.2, 4 models in memory)")

for step in range(1, PPO_STEPS + 1):
    idx = torch.randint(0, len(PROMPTS), (4,))

    loss, metrics = ppo_step(
        ppo_policy, ref_model,
        prompt_inputs["input_ids"][idx],
        prompt_inputs["attention_mask"][idx]
    )

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(ppo_policy.parameters(), 1.0)
    optimizer.step()

    ppo_metrics.append(metrics)

    if step % 5 == 0:
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"PPO Step {step}/{PPO_STEPS} | Loss: <b>{metrics['loss']:.4f}</b> | "
                 f"Reward: {metrics['reward']:.3f} | Clip: {metrics['clip_fraction']:.2f}")

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"PPO complete! Final reward: <b>{ppo_metrics[-1]['reward']:.3f}</b>")
bus.show()

In [ ]:
# 📊 PPO Training Curves
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("PPO Training Results", fontsize=14, fontweight="bold")

steps = range(1, len(ppo_metrics) + 1)
axes[0].plot(steps, [m["policy_loss"] for m in ppo_metrics], color="#7c3aed", linewidth=2)
axes[0].set_title("Policy Loss"); axes[0].grid(alpha=0.3)

axes[1].plot(steps, [m["value_loss"] for m in ppo_metrics], color="#3b82f6", linewidth=2)
axes[1].set_title("Value Loss"); axes[1].grid(alpha=0.3)

axes[2].plot(steps, [m["reward"] for m in ppo_metrics], color="#34d399", linewidth=2)
axes[2].set_title("Reward"); axes[2].grid(alpha=0.3)

axes[3].plot(steps, [m["clip_fraction"] for m in ppo_metrics], color="#f59e0b", linewidth=2)
axes[3].set_title("Clip Fraction"); axes[3].grid(alpha=0.3)

for ax in axes: ax.set_xlabel("Step")
plt.tight_layout()
plt.show()

### 4.4 SPO — Self-Play Optimization

**Key Idea:** The model competes against a previous version of itself. Responses are ranked, and the model learns to prefer winning responses. Tracks ELO ratings.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SPO — Self-Play Optimization
# Paper: Wu et al., 2024
# Key: Model improves by competing against itself, ELO tracking
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>SPO</b> training — Self-Play Optimization")

del ppo_policy, optimizer
gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

spo_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
opponent = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
opponent.eval()
optimizer = torch.optim.AdamW(spo_model.parameters(), lr=5e-5)

SPO_ROUNDS = 20
spo_metrics = []
elo_current = 1000.0
elo_history = [1000.0]

bus.send("Trainer", "broadcast", MessageType.STATUS,
         f"SPO: {SPO_ROUNDS} self-play rounds, starting ELO: {elo_current:.0f}")

for round_num in range(1, SPO_ROUNDS + 1):
    idx = torch.randint(0, len(PROMPTS), (4,))
    batch_ids = prompt_inputs["input_ids"][idx]
    batch_mask = prompt_inputs["attention_mask"][idx]

    # Generate from current model and opponent
    spo_model.eval()
    with torch.no_grad():
        current_gen = spo_model.generate(
            batch_ids, attention_mask=batch_mask,
            max_new_tokens=32, temperature=0.8, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        opponent_gen = opponent.generate(
            batch_ids, attention_mask=batch_mask,
            max_new_tokens=32, temperature=0.8, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Judge: which response is better?
    current_texts = tokenizer.batch_decode(current_gen[:, batch_ids.shape[1]:], skip_special_tokens=True)
    opponent_texts = tokenizer.batch_decode(opponent_gen[:, batch_ids.shape[1]:], skip_special_tokens=True)

    current_rewards = [simple_reward(t) for t in current_texts]
    opponent_rewards = [simple_reward(t) for t in opponent_texts]
    wins = sum(1 for c, o in zip(current_rewards, opponent_rewards) if c > o)
    win_rate = wins / len(current_rewards)

    # Update ELO
    expected = 1 / (1 + 10 ** ((1000 - elo_current) / 400))
    elo_current += 32 * (win_rate - expected)
    elo_history.append(elo_current)

    # DPO-style loss: prefer winning responses
    spo_model.train()
    winner_ids = current_gen if win_rate >= 0.5 else opponent_gen
    loser_ids = opponent_gen if win_rate >= 0.5 else current_gen

    # Pad to same length
    max_len = max(winner_ids.shape[1], loser_ids.shape[1])
    if winner_ids.shape[1] < max_len:
        winner_ids = F.pad(winner_ids, (0, max_len - winner_ids.shape[1]), value=tokenizer.eos_token_id)
    if loser_ids.shape[1] < max_len:
        loser_ids = F.pad(loser_ids, (0, max_len - loser_ids.shape[1]), value=tokenizer.eos_token_id)

    w_lp = compute_log_probs(spo_model, winner_ids, winner_ids)
    l_lp = compute_log_probs(spo_model, loser_ids, loser_ids)
    loss = -F.logsigmoid(0.1 * (w_lp - l_lp)).mean()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(spo_model.parameters(), 1.0)
    optimizer.step()

    metrics = {
        "loss": loss.item(),
        "win_rate": win_rate,
        "elo": elo_current,
        "reward": np.mean(current_rewards),
    }
    spo_metrics.append(metrics)

    # Update opponent every 5 rounds
    if round_num % 5 == 0:
        opponent.load_state_dict(spo_model.state_dict())
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"SPO Round {round_num}/{SPO_ROUNDS} | Win: <b>{win_rate:.0%}</b> | "
                 f"ELO: <b>{elo_current:.0f}</b> | Loss: {loss.item():.4f}")

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"SPO complete! Final ELO: <b>{elo_current:.0f}</b> (started at 1000)")
bus.show()

In [ ]:
# 📊 SPO Results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("SPO Self-Play Results", fontsize=14, fontweight="bold")

axes[0].plot(elo_history, color="#7c3aed", linewidth=2)
axes[0].axhline(y=1000, color="gray", linestyle="--", alpha=0.5)
axes[0].set_title("ELO Rating"); axes[0].set_xlabel("Round"); axes[0].grid(alpha=0.3)

axes[1].plot([m["win_rate"] for m in spo_metrics], color="#34d399", linewidth=2)
axes[1].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
axes[1].set_title("Win Rate vs Opponent"); axes[1].set_xlabel("Round"); axes[1].grid(alpha=0.3)

axes[2].plot([m["loss"] for m in spo_metrics], color="#f59e0b", linewidth=2)
axes[2].set_title("Self-Play Loss"); axes[2].set_xlabel("Round"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. 🔷 Advanced Techniques — Quick Demos

### 5.1 KTO — Kahneman-Tversky Optimization

**Key Idea:** Only needs binary feedback (good/bad), not paired preferences. Uses asymmetric loss inspired by prospect theory.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# KTO — Kahneman-Tversky Optimization
# Paper: Ethayarajh et al., 2024
# Key: Binary feedback only (thumbs up/down), not paired!
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>KTO</b> — works with unpaired binary feedback")

del spo_model, opponent, optimizer
gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

kto_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(kto_model.parameters(), lr=5e-5)

KTO_STEPS = 30
kto_metrics = []

for step in range(1, KTO_STEPS + 1):
    # Binary feedback: chosen = desirable, rejected = undesirable
    idx = torch.randint(0, len(chosen_ids), (4,))

    # Desirable examples
    pi_d = compute_log_probs(kto_model, chosen_ids[idx], chosen_ids[idx])
    with torch.no_grad():
        ref_d = compute_log_probs(ref_model, chosen_ids[idx], chosen_ids[idx])
    log_ratio_d = pi_d - ref_d

    # Undesirable examples
    pi_u = compute_log_probs(kto_model, rejected_ids[idx], rejected_ids[idx])
    with torch.no_grad():
        ref_u = compute_log_probs(ref_model, rejected_ids[idx], rejected_ids[idx])
    log_ratio_u = pi_u - ref_u

    # KL estimate
    kl = (torch.exp(log_ratio_d) - 1 - log_ratio_d).mean()

    # Asymmetric KTO loss
    beta = 0.1
    desirable_loss = -F.logsigmoid(beta * (log_ratio_d - kl)).mean()
    undesirable_loss = -F.logsigmoid(-beta * (log_ratio_u - kl)).mean()

    # Weight undesirable more (loss aversion from prospect theory!)
    loss = 1.0 * desirable_loss + 1.33 * undesirable_loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(kto_model.parameters(), 1.0)
    optimizer.step()

    kto_metrics.append({"loss": loss.item(), "kl": kl.item(),
                        "desirable_loss": desirable_loss.item(),
                        "undesirable_loss": undesirable_loss.item()})

    if step % 10 == 0:
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"KTO Step {step}/{KTO_STEPS} | Loss: <b>{loss.item():.4f}</b> | KL: {kl.item():.4f}")

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"KTO complete! Final loss: <b>{kto_metrics[-1]['loss']:.4f}</b>")

del kto_model, optimizer
gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

bus.show()

---

## 6. 📊 Grand Comparison — All Techniques

Let's compare everything we've trained!

In [ ]:
bus.send("Evaluator", "broadcast", MessageType.EVALUATION,
         "Generating comprehensive technique comparison...")

# Collect final metrics from each technique
comparison = {
    "DPO": {
        "final_loss": dpo_metrics[-1]["loss"],
        "steps": len(dpo_metrics),
        "key_metric": f"Accuracy: {dpo_metrics[-1]['accuracy']:.2f}",
        "color": "#7c3aed",
        "category": "Preference",
        "needs_reward_model": "No",
        "needs_ref_model": "Yes",
    },
    "GRPO": {
        "final_loss": grpo_metrics[-1]["loss"],
        "steps": len(grpo_metrics),
        "key_metric": f"Best reward: {grpo_metrics[-1]['best_reward']:.3f}",
        "color": "#3b82f6",
        "category": "RL (value-free)",
        "needs_reward_model": "Yes (or rule-based)",
        "needs_ref_model": "Yes",
    },
    "PPO": {
        "final_loss": ppo_metrics[-1]["loss"],
        "steps": len(ppo_metrics),
        "key_metric": f"Reward: {ppo_metrics[-1]['reward']:.3f}",
        "color": "#06b6d4",
        "category": "RL (full)",
        "needs_reward_model": "Yes",
        "needs_ref_model": "Yes + Value head",
    },
    "SPO": {
        "final_loss": spo_metrics[-1]["loss"],
        "steps": len(spo_metrics),
        "key_metric": f"ELO: {elo_history[-1]:.0f}",
        "color": "#34d399",
        "category": "Self-play",
        "needs_reward_model": "No",
        "needs_ref_model": "No (uses opponent)",
    },
    "KTO": {
        "final_loss": kto_metrics[-1]["loss"],
        "steps": len(kto_metrics),
        "key_metric": f"KL: {kto_metrics[-1]['kl']:.4f}",
        "color": "#f59e0b",
        "category": "Binary feedback",
        "needs_reward_model": "No",
        "needs_ref_model": "Yes",
    },
}

# Comparison table
html = '<div style="font-family:sans-serif">'
html += '<h3 style="margin-bottom:16px">📊 Technique Comparison</h3>'
html += '<table style="border-collapse:collapse;width:100%">'
html += '<tr style="border-bottom:2px solid #30363d;background:#161b22">'
for col in ["Technique", "Category", "Final Loss", "Key Metric", "Reward Model?", "Ref Model?"]:
    html += f'<th style="padding:10px 12px;text-align:left;font-size:13px">{col}</th>'
html += '</tr>'

for name, data in comparison.items():
    html += '<tr style="border-bottom:1px solid #21262d">'
    html += f'<td style="padding:8px 12px"><span style="color:{data["color"]};font-weight:700">{name}</span></td>'
    html += f'<td style="padding:8px 12px;font-size:13px">{data["category"]}</td>'
    html += f'<td style="padding:8px 12px;font-family:monospace">{data["final_loss"]:.4f}</td>'
    html += f'<td style="padding:8px 12px;font-size:13px">{data["key_metric"]}</td>'
    html += f'<td style="padding:8px 12px;font-size:13px">{data["needs_reward_model"]}</td>'
    html += f'<td style="padding:8px 12px;font-size:13px">{data["needs_ref_model"]}</td>'
    html += '</tr>'
html += '</table></div>'

display(HTML(html))

In [ ]:
# 📊 Grand Comparison Charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("All Techniques — Training Comparison", fontsize=14, fontweight="bold")

# Loss curves
for name, data in comparison.items():
    if name == "DPO":
        losses = [m["loss"] for m in dpo_metrics]
    elif name == "GRPO":
        losses = [m["loss"] for m in grpo_metrics]
    elif name == "PPO":
        losses = [m["loss"] for m in ppo_metrics]
    elif name == "SPO":
        losses = [m["loss"] for m in spo_metrics]
    elif name == "KTO":
        losses = [m["loss"] for m in kto_metrics]
    # Normalize steps to 0-1 for comparison
    x = np.linspace(0, 1, len(losses))
    axes[0].plot(x, losses, color=data["color"], linewidth=2, label=name)

axes[0].set_title("Loss Curves (normalized steps)")
axes[0].set_xlabel("Training Progress")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Final loss comparison bar chart
names = list(comparison.keys())
losses = [comparison[n]["final_loss"] for n in names]
colors = [comparison[n]["color"] for n in names]
bars = axes[1].barh(names, losses, color=colors, height=0.6)
axes[1].set_title("Final Loss (lower is better)")
axes[1].set_xlabel("Loss")
for bar, loss in zip(bars, losses):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{loss:.4f}", va="center", fontsize=10)
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

---

## 7. ⚡ Optimization Agent — Quantization

The Optimization Agent compresses the trained model for efficient deployment.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTIMIZATION — 4-bit Quantization with bitsandbytes
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "Optimizer", MessageType.COORDINATION,
         "Starting optimization stage — quantize the model")

from transformers import BitsAndBytesConfig

bus.send("Optimizer", "broadcast", MessageType.OPTIMIZATION,
         "Loading model with <b>NF4 4-bit quantization</b> (bitsandbytes)")

# Quantize with NF4
try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    quantized_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )

    # Compare sizes
    orig_params = sum(p.numel() * p.element_size() for p in ref_model.parameters()) / 1e6
    quant_params = sum(p.numel() * (0.5 if p.dtype == torch.uint8 else p.element_size()) for p in quantized_model.parameters()) / 1e6

    bus.send("Optimizer", "broadcast", MessageType.OPTIMIZATION,
             f"Original: {orig_params:.1f} MB → Quantized: ~{orig_params/4:.1f} MB (<b>~4x compression</b>)")

    # Quick quality check — generate from both
    test_prompt = "Explain machine learning in one sentence:"
    test_input = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

    ref_model.eval()
    with torch.no_grad():
        orig_out = ref_model.generate(**test_input, max_new_tokens=40, temperature=0.7, do_sample=True,
                                      pad_token_id=tokenizer.eos_token_id)
        quant_out = quantized_model.generate(**test_input, max_new_tokens=40, temperature=0.7, do_sample=True,
                                              pad_token_id=tokenizer.eos_token_id)

    bus.send("Optimizer", "broadcast", MessageType.RESULT,
             f"Quality check — Original: <i>{tokenizer.decode(orig_out[0], skip_special_tokens=True)[:100]}</i>")
    bus.send("Optimizer", "broadcast", MessageType.RESULT,
             f"Quality check — Quantized: <i>{tokenizer.decode(quant_out[0], skip_special_tokens=True)[:100]}</i>")

    del quantized_model
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

    bus.send("Optimizer", "Coordinator", MessageType.RESULT,
             "Quantization demo complete! 4-bit NF4 achieved ~4x compression ✓")
except Exception as e:
    bus.send("Optimizer", "Coordinator", MessageType.STATUS,
             f"Quantization requires GPU with bitsandbytes. Skipping. ({e})")

bus.show()

---

## 8. 📈 Full Agent Communication Log

Here's the complete record of all agent-to-agent communication throughout this session:

In [ ]:
# Show full communication log
bus.show()
print()
bus.summary_table()

---

## 9. 🧠 Understanding Guide — When to Use What

### Decision Tree for Technique Selection

In [ ]:
guide = """
<div style="font-family:sans-serif;max-width:800px;line-height:1.8">
<h2>🧠 When to Use Each Technique</h2>

<div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:24px;margin:16px 0">
<h3 style="color:#c084fc">Have paired preference data? (chosen/rejected)</h3>
<ul>
<li><b style="color:#7c3aed">DPO</b> — Simplest option. One-stage training, no RL instability.</li>
<li><b style="color:#f59e0b">SimPO</b> — Like DPO but no reference model needed.</li>
<li><b style="color:#06b6d4">IPO</b> — DPO with better regularization (if DPO overfits).</li>
</ul>
</div>

<div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:24px;margin:16px 0">
<h3 style="color:#79c0ff">Have a reward model or reward function?</h3>
<ul>
<li><b style="color:#3b82f6">GRPO</b> — Best choice for most cases. No value model = 50% less memory. Used in DeepSeek-R1.</li>
<li><b style="color:#06b6d4">PPO</b> — Full RLHF. More control but 4 models in memory. Proven at scale.</li>
<li><b style="color:#c084fc">RLHF</b> — Train reward model first, then PPO. The classic pipeline.</li>
</ul>
</div>

<div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:24px;margin:16px 0">
<h3 style="color:#56d364">Only have binary feedback? (thumbs up/down)</h3>
<ul>
<li><b style="color:#f59e0b">KTO</b> — Works with unpaired good/bad examples. No preference pairs needed!</li>
</ul>
</div>

<div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:24px;margin:16px 0">
<h3 style="color:#e3b341">Want self-improvement without external data?</h3>
<ul>
<li><b style="color:#34d399">SPO</b> — Self-play: model competes against itself.</li>
<li><b style="color:#06b6d4">SPIN</b> — Distinguish human text from model text.</li>
<li><b style="color:#c084fc">RLAIF</b> — Use AI (the model itself) as the judge.</li>
</ul>
</div>

<div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:24px;margin:16px 0">
<h3 style="color:#f87171">Memory constrained?</h3>
<ul>
<li><b style="color:#3b82f6">GRPO</b> — No value model needed (saves ~50% vs PPO).</li>
<li><b style="color:#f59e0b">SimPO / ORPO</b> — No reference model needed.</li>
<li>Combine with <b>NF4 quantization</b> for QLoRA-style training.</li>
</ul>
</div>
</div>
"""
display(HTML(guide))

In [ ]:
# Final summary
print("=" * 60)
print("🎉 AGENTIC POST-TRAINING DEMO COMPLETE!")
print("=" * 60)
print(f"\n  Techniques demonstrated: {len(comparison)}")
print(f"  Agent messages exchanged: {len(bus.history)}")
print(f"  Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"  Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"\n  Framework: github.com/sugeerth/agentic-post-training")
print("=" * 60)